In [2]:
!pip install transformers pandas torch

In [5]:
import pandas as pd
import os
import torch
from transformers import pipeline

from google.colab import drive
drive.mount('/content/drive')

def generate_colab_finbert_scores():
    print("--- Starting FinBERT Sentiment Pipeline on Google Colab ---")

    # Colab default paths
    input_file = "/content/drive/MyDrive/Colab_Notebooks/cotton_sentiment_raw.csv"
    output_path = "/content/drive/MyDrive/Colab_Notebooks/cotton_finbert_daily_scores.csv"

    if not os.path.exists(input_file):
        print(f"Error: Could not find {input_file}. Did you upload it to the left sidebar?")
        return

    # Load data
    df = pd.read_csv(input_file)
    df['Date'] = pd.to_datetime(df['Date'])

    # Initialize FinBERT on the cloud GPU
    device = 0 if torch.cuda.is_available() else -1
    print(f"Loading ProsusAI/finbert... (Using {'GPU' if device == 0 else 'CPU'})")

    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device, top_k=None)

    # Process the headlines
    print(f"Scoring {len(df)} headlines. The T4 GPU will handle this quickly...")

    pos_scores, neg_scores, neu_scores = [], [], []

    for text in df['Text']:
        try:
            # Score text (truncated to 512 tokens to prevent sequence length errors)
            result = finbert(str(text)[:512])[0]

            pos = next(item['score'] for item in result if item['label'] == 'positive')
            neg = next(item['score'] for item in result if item['label'] == 'negative')
            neu = next(item['score'] for item in result if item['label'] == 'neutral')

            pos_scores.append(pos)
            neg_scores.append(neg)
            neu_scores.append(neu)
        except Exception as e:
            pos_scores.append(0)
            neg_scores.append(0)
            neu_scores.append(1)

    # Add scores to dataframe
    df['FinBERT_Positive'] = pos_scores
    df['FinBERT_Negative'] = neg_scores
    df['FinBERT_Neutral'] = neu_scores

    # Aggregate daily averages
    daily_sentiment = df.groupby('Date')[['FinBERT_Positive', 'FinBERT_Negative', 'FinBERT_Neutral']].mean().reset_index()
    daily_sentiment['Net_Sentiment'] = daily_sentiment['FinBERT_Positive'] - daily_sentiment['FinBERT_Negative']
    daily_sentiment = daily_sentiment.sort_values('Date')

    # Export
    daily_sentiment.to_csv(output_path, index=False)

    print(f"\nSuccess! Daily scores exported to: {output_path}")
    print(daily_sentiment.head())

generate_colab_finbert_scores()

Mounted at /content/drive
--- Starting FinBERT Sentiment Pipeline on Google Colab ---
Loading ProsusAI/finbert... (Using GPU)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Scoring 1082 headlines. The T4 GPU will handle this quickly...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



Success! Daily scores exported to: /content/drive/MyDrive/Colab_Notebooks/cotton_finbert_daily_scores.csv
        Date  FinBERT_Positive  FinBERT_Negative  FinBERT_Neutral  \
0 2010-04-07          0.033055          0.053809         0.913136   
1 2010-04-26          0.015683          0.942709         0.041609   
2 2010-06-20          0.170614          0.114949         0.714437   
3 2010-10-15          0.139799          0.547295         0.312906   
4 2010-10-27          0.139322          0.049208         0.811470   

   Net_Sentiment  
0      -0.020754  
1      -0.927026  
2       0.055665  
3      -0.407496  
4       0.090115  
